# Intro to Pseudorandom Number Generation

Don't try this at home!

Seriously though, this is just to give you an idea of how PRNGs work, specifically:

- How an integer can be converted into an "internal" state
- How that internal state can be manipulated to generate apparent randomness

The class below implements a 4-bit **Linear-Feedback Shift Register (LFSR)**. It is a super-basic PRNG with a very short period of $2^4 - 1 = 15$ and not good for any real work.

See also: [https://en.wikipedia.org/wiki/Linear-feedback_shift_register](https://en.wikipedia.org/wiki/Linear-feedback_shift_register)

You can compare the complexity of the LFSR algorithm to something a bit more modern, like the [Mersenne Twister](https://en.wikipedia.org/wiki/Mersenne_Twister).

---

**Fun Fact**: Super Mario Bros 3 (1988), and many other early console games, implemented LFSRs as very lightweight and "good-enough" random number generators. SMB3 in particular used a 15-bit variant.

---

In [ ]:
class LFSR4:
    def __init__(self, seed: int):
        # Ensure the seed is between 1 and 15
        seed = (seed % 15) + 1

        # Convert the integer seed to 4 bits
        # and store as the internal state
        self.state = [int(bit) for bit in f'{seed:04b}']
        return

    def step(self) -> int:
        '''
        Generate one bit and update the internal state.
        '''
        # We will output the rightmost bit
        output = self.state[-1]

        # XOR the two feedback bits
        new_bit = self.state[0] ^ self.state[-1]

        # Shift everything right and insert the new bit on the left
        self.state = [new_bit] + self.state[:-1]
        return output

    def generate(self, n: int) -> list[int]:
        '''
        Generate n bits.
        '''
        return [self.step() for _ in range(n)]

In [ ]:
# Initialize the rng
seed = 42
rng = LFSR4(seed)

In [ ]:
# Generate some random bits
rng.generate(10)

In [ ]:
# The internal state at this point
rng.state

In [ ]:
# Generate one additional bit
rng.generate(1)

In [ ]:
# Notice how the state has shifted to the right with a new bit at the beginning
rng.state

In [ ]:
# Generate a bunch of bits
bit_stream = rng.generate(90)

# This PRNG has a period of 2^4 - 1
period = 2**4 - 1

# Notice the repetition!
for start in range(0, len(bit_stream), period):
    end = start + period
    label = f'Bits {start + 1} through {end}:'
    
    print(label)
    print('\t', bit_stream[start:end], '\n')

---

Just for a bit more fun, and because someone in class recently mentioned assembly code, here's a link to the [6502](https://en.wikipedia.org/wiki/MOS_Technology_6502) code for SMB3: https://raw.githubusercontent.com/captainsouthbird/smb3/master/PRG/prg030.asm

Search for 'Randomize' to find the code for the LFSR implementation. 

Below is a version that ChatGPT cleaned up for me:

```
Randomize:
    LDX #$00          ; Start with byte 0
    LDY #$09          ; There are 9 bytes of state

    LDA rng_bytes
    AND #$02          ; Extract bit 1 of byte 0
    STA scratch

    LDA rng_bytes+1
    AND #$02          ; Extract bit 1 of byte 1
    EOR scratch       ; XOR the two tap bits

    CLC               ; Assume feedback bit = 0
    BEQ Rotate
    SEC               ; If XOR != 0, feedback bit = 1

Rotate:
    ROR rng_bytes,X   ; Rotate right through carry
    INX
    DEY
    BNE Rotate

    RTS
```